In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from copy import deepcopy

from src.dataloaders_2 import (
    TaskILMNIST, ClassILMNIST5Task,
    TaskILCIFAR10, ClassILCIFAR5Task,
    TaskILTinyImageNet, ClassILTinyImageNet10Task
)
from src.utils import dotdict
from torch.utils.data import DataLoader, Subset

from tqdm import tqdm
import os

# Base config - will be modified per dataloader test
base_config = {
    "lr": 2e-4,
    "batch_size": 256,
    "epochs": 20,
    "num_workers": 0,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "seed": 0,
    "flatten_imgs": "default",  # Let each dataloader decide
    "use_cnn_encoder": True,
    "cnn_encoder": "resnet18",
    "cnn_pretrained": True,
    "encoder_freeze": True,
}

def test_dataloader(dataloader_fn, name, expected_tasks, expected_classes_per_task, config_overrides=None):
    """Test a dataloader and verify task/class structure."""
    print(f"\n{'='*60}")
    print(f"Testing: {name}")
    print(f"Expected: {expected_tasks} tasks, {expected_classes_per_task} classes/task")
    print(f"{'='*60}")
    
    # Create config
    cfg = dotdict({**base_config})
    cfg.num_tasks = expected_tasks
    cfg.classes_per_task = expected_classes_per_task
    if config_overrides:
        for k, v in config_overrides.items():
            cfg[k] = v
    
    try:
        # Initialize dataloader
        dl = dataloader_fn(cfg)
        print(f"✓ Dataloader initialized")
        print(f"  - num_tasks: {dl.num_tasks}")
        print(f"  - classes_per_task: {dl.classes_per_task}")
        print(f"  - flatten: {dl.flatten}")
        print(f"  - tasks: {dl.tasks[:3]}..." if len(dl.tasks) > 3 else f"  - tasks: {dl.tasks}")
        
        # Test each task
        for task_id in range(dl.num_tasks):
            train_loader, test_loader = dl.get_dataloaders(task_id)
            
            # Get a sample batch
            x_train, y_train = next(iter(train_loader))
            x_test, y_test = next(iter(test_loader))
            
            # Check output dimensions
            train_classes = y_train.shape[1]
            test_classes = y_test.shape[1]
            
            # For ClassIL, test set grows; for TaskIL, it stays at classes_per_task
            print(f"  Task {task_id}: train={len(train_loader.dataset)}, test={len(test_loader.dataset)}, "
                  f"input_shape={list(x_train.shape[1:])}, train_output_classes={train_classes}, test_output_classes={test_classes}")
        
        print(f"✓ All {dl.num_tasks} tasks passed!")
        return True
        
    except Exception as e:
        print(f"✗ FAILED: {e}")
        import traceback
        traceback.print_exc()
        return False

# Run all tests
print("DATALOADER VALIDATION TESTS")
print("="*60)

results = {}

# MNIST tests
results["TaskILMNIST"] = test_dataloader(
    TaskILMNIST, "TaskIL MNIST", 
    expected_tasks=5, expected_classes_per_task=2
)

results["ClassILMNIST5Task"] = test_dataloader(
    ClassILMNIST5Task, "ClassIL MNIST (5 tasks)", 
    expected_tasks=5, expected_classes_per_task=2
)

# # CIFAR10 tests
results["TaskILCIFAR10"] = test_dataloader(
    TaskILCIFAR10, "TaskIL CIFAR10", 
    expected_tasks=5, expected_classes_per_task=2
)

results["ClassILCIFAR5Task"] = test_dataloader(
    ClassILCIFAR5Task, "ClassIL CIFAR10 (5 tasks)", 
    expected_tasks=5, expected_classes_per_task=2
)

# TinyImageNet tests (requires data to be downloaded)
# Uncomment if you have TinyImageNet data:
results["TaskILTinyImageNet"] = test_dataloader(
    TaskILTinyImageNet, "TaskIL TinyImageNet", 
    expected_tasks=10, expected_classes_per_task=20
)
results["ClassILTinyImageNet10Task"] = test_dataloader(
    ClassILTinyImageNet10Task, "ClassIL TinyImageNet (10 tasks)", 
    expected_tasks=10, expected_classes_per_task=20
)

# Summary
print("\n" + "="*60)
print("SUMMARY")
print("="*60)
for name, passed in results.items():
    status = "✓ PASS" if passed else "✗ FAIL"
    print(f"  {name}: {status}")

DATALOADER VALIDATION TESTS

Testing: TaskIL CIFAR10
Expected: 5 tasks, 2 classes/task


/cluster/raid/home/yassine/.conda/envs/efc/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/cluster/raid/home/yassine/.conda/envs/efc/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


DataLoader using device: cuda
✓ Dataloader initialized
  - num_tasks: 5
  - classes_per_task: 2
  - flatten: True
  - tasks: [[0, 1], [2, 3], [4, 5]]...
  Task 0: train=10000, test=2000, input_shape=[512], train_output_classes=2, test_output_classes=2
  Task 1: train=10000, test=2000, input_shape=[512], train_output_classes=2, test_output_classes=2
  Task 2: train=10000, test=2000, input_shape=[512], train_output_classes=2, test_output_classes=2
  Task 3: train=10000, test=2000, input_shape=[512], train_output_classes=2, test_output_classes=2
  Task 4: train=10000, test=2000, input_shape=[512], train_output_classes=2, test_output_classes=2
✓ All 5 tasks passed!

Testing: TaskIL TinyImageNet
Expected: 10 tasks, 20 classes/task
DataLoader using device: cuda
✓ Dataloader initialized
  - num_tasks: 10
  - classes_per_task: 20
  - flatten: True
  - tasks: [[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19], [20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35,

# Legacy code

In [9]:
def evaluate(net, loader, task_id, task_classes):
    net.eval()
    correct = total = 0.0
    total_loss = 0.0
    task_set = set(range(task_classes[0], task_classes[1] + 1))
    criterion = net.loss_fn  # CE or MSE

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(config.device), y.to(config.device)
            net.task_id = task_id
            out = net(x)
            true_labels = y.argmax(1)

            mask = torch.isin(true_labels, torch.tensor(list(task_set), device=config.device))
            if not mask.any():
                continue

            out = out[mask]
            y = y[mask]
            true_labels = true_labels[mask]

            start_idx = task_classes[0]
            pred_rel = out[:, start_idx:task_classes[1]+1].argmax(1)
            pred_abs = pred_rel + start_idx

            correct += (pred_abs == true_labels).sum().item()
            total += true_labels.size(0)
            total_loss += criterion(out[:, :task_classes[1]+1], y.argmax(1)).item() * true_labels.size(0)

    acc = correct / total if total else 0.0
    loss = total_loss / total if total else 0.0
    return acc, loss

In [ ]:
bp_net_A = BP_network(config).to(config.device)
bp_net_A.task_id = 0
optimizer = optim.Adam(bp_net_A.parameters(), lr=config.lr)

bp_net_A.train()

for epoch in range(config.epochs):
    pbar = tqdm(total=len(train_loader_A), desc="BP (Task A)", unit="epoch", leave=True)
    for x, y in train_loader_A:
        x, y = x.to(config.device), y.to(config.device)

        optimizer.zero_grad()
        y_hat = bp_net_A(x)
        _ = bp_net_A.calculate_loss(y_hat, y.argmax(dim=1))
        bp_net_A.backward(y)          # BP-specific backward
        optimizer.step()

        pbar.update(1)

    # Train accuracy on Task A
    test_acc_A, _ = evaluate(bp_net_A, test_loader_A, task_id=0, task_classes=[0, 4])
    test_acc_B, _ = evaluate(bp_net_A, test_loader_B, task_id=1, task_classes=[5, 9])
    combined_test_acc, _ = evaluate(bp_net_A, test_loader_B, task_id=1, task_classes=[0, 9])  # all 10 classes are present

    # (acc_A, loss_A)       = evaluate(bp_net_A, test_loader_A, task_id=0, task_classes=[0, 4])
    # (acc_B, loss_B)       = evaluate(bp_net_A, test_loader_B, task_id=1, task_classes=[5, 9])
    # (acc_comb, loss_comb) = evaluate(bp_net_A, test_loader_B, task_id=1, task_classes=[0, 9])

    # Update the progress-bar postfix
    pbar.set_postfix({
        "Test A" : f"{test_acc_A:.3f}",
        "Test B" : f"{test_acc_B:.3f}",
        "Comb"   : f"{combined_test_acc:.3f}"
    })

    pbar.close()

BP (Task A): 100%|██████████| 120/120 [00:00<00:00, 134.96epoch/s, Test A=0.986, Test B=0.224, Comb=0.507]


In [ ]:
# Test TinyImageNet dataloader - visualize images and labels
import matplotlib.pyplot as plt
import torchvision

# Get a batch from the train loader
data_iter = iter(train_loader_A)
images, labels = next(data_iter)

print(f"Batch shape: {images.shape}")
print(f"Labels shape: {labels.shape}")
print(f"Image min/max: {images.min():.3f} / {images.max():.3f}")

# Convert one-hot labels to class indices
label_indices = labels.argmax(dim=1)
print(f"\nFirst 16 labels (class indices): {label_indices[:16].tolist()}")

# Denormalize images for visualization (using ImageNet stats)
# Move to CPU first, then denormalize with CPU tensors
images_cpu = images[:16].cpu()
mean = torch.tensor([0.485, 0.456, 0.406], device='cpu').view(3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225], device='cpu').view(3, 1, 1)
images_denorm = images_cpu * std + mean
images_denorm = torch.clamp(images_denorm, 0, 1)

# Create a grid of images
fig, axes = plt.subplots(4, 4, figsize=(10, 10))
axes = axes.flatten()

for idx in range(16):
    img = images_denorm[idx].permute(1, 2, 0).numpy()
    axes[idx].imshow(img)
    axes[idx].set_title(f"Class: {label_indices[idx].item()}", fontsize=10)
    axes[idx].axis('off')

plt.tight_layout()
plt.suptitle("TinyImageNet Sample Images (64x64)", fontsize=14, y=1.00)
plt.show()

# Verify unique classes in the batch
unique_classes = torch.unique(label_indices)
print(f"\nUnique classes in this batch: {unique_classes.tolist()}")
print(f"Total unique classes: {len(unique_classes)}")

# Check a few batches to see class distribution
print("\n--- Checking first 5 batches ---")
data_iter = iter(train_loader_A)
for batch_idx in range(5):
    _, labels_batch = next(data_iter)
    label_indices_batch = labels_batch.argmax(dim=1)
    unique_batch = torch.unique(label_indices_batch)
    print(f"Batch {batch_idx}: {len(label_indices_batch)} samples, "
          f"Classes: {unique_batch.tolist()}")